# Advanced Retrieval

**Module:** 04 — RAG

Go beyond naive top-k vectors: hybrid search, parent-child indexes, multi-query, compression, metadata filters, and time-aware retrieval.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Combine BM25 and vectors with fusion
- Design parent-child indexing for precision + context
- Expand queries and compress contexts deliberately
- Apply metadata and time filters safely


## Hybrid BM25 + Vector

**Definition.** **Hybrid retrieval** fuses lexical scores (BM25/keyword) with dense vector similarity so exact terms and paraphrases both win.

**Why it matters.** IDs, error codes, and rare proper nouns crush pure dense retrieval.

**How it works.** Run both retrievers; fuse with RRF or weighted score normalization.

**Intuition.** Dictionary lookup + semantic search in one ranked list.

**Common pitfalls.**
- Uncalibrated score mixing
- Ignoring analyzers/tokenization for BM25

**When to use.** Default for enterprise search/RAG.

| Method | Captures | Misses |
|--------|----------|--------|
| BM25 | Exact terms, IDs | Paraphrase |
| Vector | Paraphrase | Rare tokens |
| Hybrid | Both | Needs fusion tuning |


In [ ]:
# Demo 1 — toy BM25-ish score
import math
from collections import Counter

def bm25_score(query, doc, k1=1.2, b=0.75, avgdl=5):
    q, d = query.lower().split(), doc.lower().split()
    tf = Counter(d); score = 0.0; dl = len(d)
    for term in q:
        f = tf[term]
        if f == 0: continue
        idf = math.log(1 + 10)  # toy N
        score += idf * (f * (k1+1)) / (f + k1*(1-b + b*dl/avgdl))
    return score
print(bm25_score("error E42 refund", "refund policy for error E42 cases"))
print(bm25_score("error E42 refund", "shipping timelines for parcels"))


In [ ]:
# Demo 2 — reciprocal rank fusion
def rrf(rank_lists, k=60):
    scores = {}
    for lst in rank_lists:
        for rank, doc_id in enumerate(lst, 1):
            scores[doc_id] = scores.get(doc_id, 0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: -x[1])
print(rrf([["A","B","C"], ["B","C","D"]]))


In [ ]:
# Demo 3 — hybrid API-ish payload
import json
print(json.dumps({
  "query": "error E42 refund",
  "vector_k": 40, "bm25_k": 40, "fusion": "rrf", "out_k": 10
}, indent=2))


### Try it yourself — Hybrid BM25 + Vector

1. Implement RRF over three rank lists.
2. Find a query where BM25 beats vectors in your domain.
3. Tune k in RRF and observe reordering.


## Parent-Child Indexing

**Definition.** Index **small child chunks** for precise matching but return **larger parent** sections to the LLM for context.

**Why it matters.** Tiny chunks retrieve well; large chunks read well—use both.

**How it works.** Store parent_id on children; at query time map top children → unique parents.

**Intuition.** Find the sentence, quote the section.

**Common pitfalls.**
- Parents too huge for the window
- Orphan children without parents

**When to use.** Manuals, long policies, books.


In [ ]:
# Demo 1 — parent-child structures
parents = {"P1": "Refund Policy\n... full section ... 60 days ... exceptions ..."}
children = [
    {"id": "C1", "parent": "P1", "text": "Refunds within 60 days"},
    {"id": "C2", "parent": "P1", "text": "Exceptions for clearance items"},
]
hits = ["C2", "C1"]
parent_ids = list(dict.fromkeys(children[int(h[1])-1]["parent"] for h in hits))
print("parents to pack:", parent_ids)
print(parents[parent_ids[0]][:60])


In [ ]:
# Demo 2 — expand unique parents with budget
def expand(child_hits, child_to_parent, parent_text, budget=2):
    out = []
    for cid in child_hits:
        pid = child_to_parent[cid]
        if pid not in out:
            out.append(pid)
        if len(out) >= budget: break
    return [(pid, parent_text[pid]) for pid in out]
print(expand(["C1","C2","C1"], {"C1":"P1","C2":"P1"}, {"P1": "full section"}))


### Try it yourself — Parent-Child Indexing

1. Design child size vs parent size for a 20-page PDF.
2. Handle multiple children mapping to the same parent without dupes.


## Multi-Query Retrieval

**Definition.** Generate several query variants, retrieve per variant, then fuse results.

**Why it matters.** One wording rarely matches how docs were written.

**How it works.** LLM/rules expand → parallel retrieve → RRF/dedupe.

**Intuition.** Ask the librarian three ways.

**Common pitfalls.**
- Variant drift
- Cost multiplication

**When to use.** Recall-sensitive FAQ and diverse phrasing.


In [ ]:
# Demo 1 — variants + fuse
def variants(q):
    return [q, f"policy on {q}", f"how do I {q}"]
def retrieve(q):  # fake ranks by words
    return [w for w in q.split() if len(w) > 3][:3] or ["doc"]
from collections import Counter
scores = Counter()
for v in variants("reset password"):
    for i, doc in enumerate(retrieve(v), 1):
        scores[doc] += 1/(60+i)
print(scores.most_common())


In [ ]:
# Demo 2 — budget: cap expansions
MAX_VARIANTS = 3
vs = variants("international shipping duties")[:MAX_VARIANTS]
print(len(vs), vs)


### Try it yourself — Multi-Query Retrieval

1. Write 5 manual variants for a domain query and fuse toy ranks.


## Contextual Compression

**Definition.** Shrink retrieved text to spans that actually answer the question before LLM packing.

**Why it matters.** Extra tokens cost money and dilute attention.

**How it works.** Extractive filters, embedding sentence filters, or LLM compressors.

**Intuition.** Highlighter, not photocopier.

**Common pitfalls.**
- Over-compression removing constraints
- Latency of LLM compressors

**When to use.** Long parents or noisy web pages.


In [ ]:
# Demo 1 — sentence filter by overlap
import re
def compress(query, text, keep=2):
    q = set(query.lower().split())
    sents = re.split(r"(?<=[.!?])\s+", text)
    ranked = sorted(sents, key=lambda s: len(q & set(s.lower().split())), reverse=True)
    return " ".join(ranked[:keep])
doc = "We ship worldwide. Refunds within 60 days. Password resets are online. Cookies taste good."
print(compress("refund policy", doc))


In [ ]:
# Demo 2 — token budget compression metric
orig, comp = 400, 90
print({"saved_frac": 1 - comp/orig, "kept_tokens": comp})


### Try it yourself — Contextual Compression

1. Compress a parent section to ≤50 words without losing the numeric answer.


## Metadata Filtering

**Definition.** Constrain retrieval with structured predicates (tenant, product, lang, ACL).

**Why it matters.** Security and relevance both depend on filters.

**How it works.** Prefilter in the DB query or post-filter candidates before packing.

**Intuition.** Only search the shelves you may enter.

**Common pitfalls.**
- Filter/retrieval mismatch empty results
- Missing fields → silent drops

**When to use.** Multi-tenant and multi-product systems—mandatory.


In [ ]:
# Demo 1 — prefilter
rows = [{"id":1,"tenant":"acme","lang":"en"},{"id":2,"tenant":"acme","lang":"de"},
        {"id":3,"tenant":"beta","lang":"en"}]
def prefilter(rows, **pred):
    return [r for r in rows if all(r.get(k)==v for k,v in pred.items())]
print(prefilter(rows, tenant="acme", lang="en"))


In [ ]:
# Demo 2 — deny by default ACL
def allowed(chunk_acl, user_roles):
    return bool(set(chunk_acl) & set(user_roles))
print(allowed(["role:legal"], ["role:support"]))
print(allowed(["role:support"], ["role:support"]))


### Try it yourself — Metadata Filtering

1. Write filter predicates for geo-restricted pricing docs.


## Time-based Retrieval

**Definition.** Prefer or restrict documents by recency or valid-time intervals.

**Why it matters.** Policies supersede; outdated chunks cause compliance errors.

**How it works.** Filter modified_at/valid_to; or boost freshness in ranking.

**Intuition.** Yesterday's policy may be wrong today.

**Common pitfalls.**
- Timezone bugs
- Boosting recency over correctness always

**When to use.** Policies, prices, release notes, news.


In [ ]:
# Demo 1 — valid-time filter
from datetime import date
docs = [
    {"id": "v1", "valid_from": date(2024,1,1), "valid_to": date(2025,12,31), "text": "30 days"},
    {"id": "v2", "valid_from": date(2026,1,1), "valid_to": date(9999,1,1), "text": "60 days"},
]
def as_of(asof):
    return [d for d in docs if d["valid_from"] <= asof <= d["valid_to"]]
print(as_of(date(2026,8,1)))


In [ ]:
# Demo 2 — freshness boost
import math
def boost(score, age_days, half_life=90):
    return score + 0.1 * math.exp(-math.log(2)*age_days/half_life)
print(boost(0.7, 10), boost(0.7, 200))


In [ ]:
# Demo 3 — combine filter + hybrid ranks
print({"filter": "valid_to >= today", "then": "hybrid retrieve", "then2": "freshness boost"})


### Try it yourself — Time-based Retrieval

1. Model a policy that changes mid-year with valid_from/valid_to.
2. Show a query that would return the wrong version without time filters.


## Glossary

- **RRF**: Reciprocal Rank Fusion
- **parent-child**: Small match units, large read units


## Summary & Key Takeaways

- Hybrid is the practical default for enterprise RAG
- Parent-child balances precision and readability
- Multi-query and compression trade cost for recall/precision
- Metadata and time filters are safety features

### Practice

Break a pure-vector baseline with an ID-heavy query; fix it with hybrid+filters.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
